# Module 24: Capstone Projects
## Complete Guide to All 5 Capstone Projects

This notebook serves as a comprehensive guide for all five capstone projects.
Each project includes an architecture diagram, implementation steps, evaluation
criteria, and key code patterns. Use this as a reference while completing your
chosen project(s).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, classification_report
from typing import List, Dict, Tuple, Optional

print("All imports ready for all 5 capstone projects")

---
## Project 1: End-to-End ML Pipeline

### Architecture
```
┌──────────┐    ┌─────────────┐    ┌──────────────────┐    ┌──────────────┐
│ Raw Data │───>│ EDA & Viz  │───>│ Feature Engineer │───>│ Train/Test   │
│ (CSV)    │    │ (pandas/seaborn)│  │ (sklearn)       │    │ Split        │
└──────────┘    └─────────────┘    └──────────────────┘    └──────┬───────┘
                                                                  │
         ┌────────────────────────────────────────────────────────┘
         v
┌──────────────────┐    ┌──────────────┐    ┌───────────────────┐
│ Model Selection  │───>│ Hyperparam   │───>│ Evaluation        │
│ (3+ models)      │    │ Tuning (CV)  │    │ (metrics + plots) │
└──────────────────┘    └──────────────┘    └──────────┬────────┘
                                                        │
                                                        v
┌──────────────────┐    ┌───────────────────────────────┐
│ Interpretation   │<───│ MLflow Tracking              │
│ (SHAP/feature    │    │ (params, metrics, artifacts) │
│  importance)     │    └───────────────────────────────┘
└──────────────────┘
```

### Implementation Steps
1. **Data Loading and Inspection**: Use pandas to load and explore the dataset
2. **EDA**: Generate summary statistics, distributions, correlations, missing values
3. **Feature Engineering**: Create new features, handle missing values, encode categories
4. **Pipeline Construction**: Build sklearn Pipeline with ColumnTransformer
5. **Model Training**: Train 3+ models (Linear, Tree-based, Ensemble)
6. **Hyperparameter Tuning**: Use GridSearchCV or RandomizedSearchCV
7. **Evaluation**: Compute metrics, plot learning curves, feature importance
8. **MLflow Tracking**: Log all parameters, metrics, and artifacts
9. **Model Interpretation**: Analyze feature importance or SHAP values

### Evaluation Criteria
- R² > 0.8 for regression / F1 > 0.85 for classification
- At least 3 models compared with cross-validation
- Complete MLflow experiment with all runs logged
- Feature importance analysis included
- Clean, documented code with modular structure

In [ ]:
# Project 1: Core pipeline pattern
def build_ml_pipeline(numerical_cols, categorical_cols):
    """Build a complete sklearn pipeline."""
    num_transformer = Pipeline([
        ("scaler", StandardScaler()),
    ])
    cat_transformer = Pipeline([
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", num_transformer, numerical_cols),
        ("cat", cat_transformer, categorical_cols),
    ])
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(random_state=42)),
    ])
    return pipeline


print("Project 1: ML Pipeline pattern ready")
print("  Use with: pipeline = build_ml_pipeline(num_cols, cat_cols)")

---
## Project 2: Data Engineering ETL Pipeline

### Architecture
```
┌──────────┐    ┌────────────────┐    ┌────────────────┐    ┌──────────────┐
│ Source 1 │───>│                │    │                │    │              │
│ (CSV)    │    │   EXTRACT     │───>│   TRANSFORM    │───>│     LOAD     │
├──────────┤    │  (pandas/     │    │  (pandas:      │    │  (SQLite)    │
│ Source 2 │───>│   requests)   │    │   clean, merge,│    │              │
│ (API)    │    │                │    │   aggregate)   │    └──────┬───────┘
└──────────┘    └────────────────┘    └────────────────┘         │
                                                                  v
┌────────────────────────────────────────────────────────────────────┐
│ Airflow DAG (scheduled daily @ 2 AM)                              │
│ extract_task >> transform_task >> load_task >> validate_task      │
└────────────────────────────────────────────────────────────────────┘
```

### Implementation Steps
1. **Extract**: Read from CSV file and/or REST API endpoint
2. **Transform**: Clean data, merge sources, create new columns, handle quality issues
3. **Load**: Write processed data to SQLite database with proper schema
4. **Validate**: Check row counts, null percentages, data types
5. **Orchestrate**: Create Airflow DAG with scheduled execution
6. **Monitor**: Add logging at each step, error handling with retries

### Evaluation Criteria
- Successfully extracts from 2+ different sources
- Transformations are correct and validated
- Data loaded into SQLite with proper schema
- Airflow DAG runs successfully on schedule
- Error handling and logging present
- Pipeline is idempotent (can run multiple times safely)

In [ ]:
# Project 2: ETL function pattern
def etl_pipeline(csv_path: str, api_url: str, db_path: str = "data/pipeline.db"):
    """Complete ETL pipeline."""
    # Extract
    print("[EXTRACT] Loading data...")
    df_csv = pd.read_csv(csv_path)
    import requests
    df_api = pd.DataFrame(requests.get(api_url).json())

    # Transform
    print("[TRANSFORM] Cleaning and merging...")
    df = df_csv.merge(df_api, on="id", how="left")
    df = df.dropna(subset=["target"])
    df["created_at"] = pd.Timestamp.now()

    # Load
    print("[LOAD] Writing to SQLite...")
    import sqlite3
    conn = sqlite3.connect(db_path)
    df.to_sql("processed_data", conn, if_exists="replace", index=False)
    conn.close()

    # Validate
    print(f"[VALIDATE] Loaded {len(df)} rows, {len(df.columns)} columns")
    return {"rows": len(df), "columns": list(df.columns)}


print("Project 2: ETL pipeline pattern ready")

---
## Project 3: Recommendation System

### Architecture
```
┌──────────────┐    ┌──────────────────┐    ┌─────────────────┐    ┌─────────────┐
│ User-Item    │───>│ Similarity       │───>│ Top-K           │───>│ Evaluation  │
│ Rating Matrix│    │ Computation      │    │ Recommendations │    │ precision@k │
│ (pivot table)│    │ (cosine/pearson) │    │                  │    │ recall@k    │
└──────────────┘    └──────────────────┘    └─────────────────┘    └─────────────┘
                                                                          
┌──────────────┐    ┌──────────────────┐                                 
│ Content      │───>│ Item Features    │───> Alternative path             
│ Data         │    │ (TF-IDF on meta) │    if collaborative fails         
└──────────────┘    └──────────────────┘                                 
```

### Implementation Steps
1. **Load Data**: Load ratings data (user_id, item_id, rating, timestamp)
2. **Explore**: Analyze rating distribution, user/item coverage, sparsity
3. **Build Matrix**: Create user-item rating matrix (pivot table)
4. **Collaborative Filtering**: Compute cosine similarity between users or items
5. **Content-Based (alt)**: Compute item similarity using metadata features
6. **Generate Recommendations**: For a given user, predict top-N items
7. **Evaluate**: Compute precision@k, recall@k, MAP for held-out ratings
8. **Handle Cold Start**: Strategy for new users or new items

### Evaluation Criteria
- precision@5 > 0.3 and recall@5 > 0.1
- Handles cold-start problem explicitly
- Both user-based and item-based CF implemented
- Evaluation on held-out test set (temporal split preferred)
- Results presented with clear explanation

In [ ]:
# Project 3: Collaborative filtering pattern
def collaborative_recommend(ratings_df, user_id, k=5, metric="cosine"):
    """Generate top-K recommendations using collaborative filtering."""
    matrix = ratings_df.pivot_table(
        index="user_id", columns="item_id", values="rating"
    ).fillna(0)
    from sklearn.metrics.pairwise import cosine_similarity
    sim_matrix = cosine_similarity(matrix)

    user_idx = matrix.index.get_loc(user_id)
    sim_scores = list(enumerate(sim_matrix[user_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:6]  # top 5 similar users

    similar_users = [matrix.index[i] for i, _ in sim_scores]
    user_rated = set(ratings_df[ratings_df["user_id"] == user_id]["item_id"])

    candidates = ratings_df[ratings_df["user_id"].isin(similar_users)]
    candidates = candidates[~candidates["item_id"].isin(user_rated)]
    recommendations = (
        candidates.groupby("item_id")["rating"].mean().nlargest(k)
    )
    return recommendations.index.tolist()


def precision_at_k(recommended, relevant, k):
    return len(set(recommended[:k]) & set(relevant)) / k


def recall_at_k(recommended, relevant, k):
    return len(set(recommended[:k]) & set(relevant)) / len(relevant) if relevant else 0


print("Project 3: Recommendation system patterns ready")

---
## Project 4: NLP Project

### Architecture
```
┌──────────┐    ┌────────────────┐    ┌─────────────────┐    ┌──────────────┐
│ Raw Text │───>│ Preprocessing  │───>│ Feature Extract │───>│ Model Train  │
│ (CSV/txt)│    │ (clean, lower, │    │ (TF-IDF,        │    │ (Logistic,   │
│          │    │  tokenize,     │    │  n-grams,       │    │  Naive Bayes,│
│          │    │  remove_stop)  │    │  embeddings)    │    │  SVM)        │
└──────────┘    └────────────────┘    └─────────────────┘    └──────┬───────┘
                                                                     │
         ┌───────────────────────────────────────────────────────────┘
         v
┌──────────────────┐    ┌──────────────────┐    ┌───────────────────┐
│ Evaluation       │───>│ Error Analysis   │───>│ sklearn Pipeline │
│ classification   │    │ (confusion       │    │ (deployable      │
│ report + metrics │    │  matrix, samples)│    │  single object)  │
└──────────────────┘    └──────────────────┘    └───────────────────┘
```

### Implementation Steps
1. **Load Data**: Load text data with labels
2. **EDA**: Analyze text length, word frequency, class balance
3. **Preprocessing**: Lowercase, remove punctuation/stopwords, lemmatize
4. **Feature Extraction**: TF-IDF with n-grams (unigram + bigram)
5. **Model Building**: Train 3 models (Naive Bayes, Logistic Regression, SVM)
6. **Pipeline**: Wrap everything in a single sklearn Pipeline
7. **Evaluation**: Classification report, confusion matrix, ROC-AUC
8. **Error Analysis**: Examine misclassified samples for patterns

### Evaluation Criteria
- F1 > 0.85 on test set
- sklearn Pipeline with preprocessing + model
- Error analysis with specific examples
- At least 3 models compared
- Proper train/test split (temporal if applicable)

In [ ]:
# Project 4: NLP pipeline pattern
def build_nlp_pipeline():
    """Build a complete NLP pipeline."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words="english",
            max_df=0.8,
            min_df=2,
        )),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ])
    return pipeline


def preprocess_text(text: str) -> str:
    """Clean and normalize text."""
    import re
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


print("Project 4: NLP pipeline patterns ready")

---
## Project 5: Production Prediction API

### Architecture
```
┌─────────────────────────────────────────────────────────────────────┐
│                           User/Client                              │
└─────────────────────────┬───────────────────────────────────────────┘
                          │
                          v
┌──────────────────────────────────────────────────────────────────────┐
│                      Docker Container                               │
│  ┌───────────────────────────────────────────────────────────────┐  │
│  │  FastAPI App (Gunicorn + Uvicorn Workers)                    │  │
│  │  ┌─────────────┐  ┌─────────────┐  ┌────────────────────┐   │  │
│  │  │ GET /       │  │ GET /health │  │ POST /predict      │   │  │
│  │  │ API Info    │  │ Health Check│  │ Pydantic Validation│   │  │
│  │  └─────────────┘  └─────────────┘  └──────────┬─────────┘   │  │
│  │                                                │            │  │
│  │                                                v            │  │
│  │                                        ┌──────────────┐    │  │
│  │                                        │ Model (joblib)│    │  │
│  │                                        └──────────────┘    │  │
│  └───────────────────────────────────────────────────────────────┘  │
└──────────────────────────────────────────────────────────────────────┘
                          │
                          v
┌──────────────────┐    ┌──────────────────┐    ┌──────────────────┐
│ pytest Tests     │    │ CI/CD (GitHub    │    │ Logging +        │
│ (TestClient)     │    │  Actions)        │    │ Monitoring       │
└──────────────────┘    └──────────────────┘    └──────────────────┘
```

### Implementation Steps
1. **Train Model**: Train and save a model with joblib
2. **FastAPI App**: Create API with Pydantic schemas and proper error handling
3. **Testing**: Write tests using TestClient for all endpoints
4. **Docker**: Create Dockerfile (multi-stage), docker-compose.yml, .dockerignore
5. **CI/CD**: GitHub Actions workflow (test, build, push, deploy)
6. **Logging**: Configure structured logging for requests and errors
7. **Health Checks**: Add /health endpoint and Docker HEALTHCHECK

### Evaluation Criteria
- API responds correctly to all endpoints
- Pydantic validation works (invalid inputs return 422)
- Docker image builds and runs successfully
- Tests pass with >80% code coverage
- CI/CD pipeline is configured and functional
- Health checks and error handling work properly

In [ ]:
# Project 5: FastAPI + Pydantic pattern
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional


class PredictInput(BaseModel):
    features: List[float] = Field(..., description="Feature vector")
    model_version: Optional[str] = Field(default="latest")

    @field_validator("features")
    def validate_features(cls, v):
        if len(v) == 0:
            raise ValueError("features cannot be empty")
        if len(v) > 100:
            raise ValueError("too many features (max 100)")
        return v


class PredictOutput(BaseModel):
    prediction: float
    model_version: str
    processing_time_ms: float


class HealthOutput(BaseModel):
    status: str
    model_loaded: bool
    uptime_seconds: float


print("Project 5: API schema patterns ready")
print("  Use with FastAPI: app = FastAPI()")

---
## Common Capstone Checklist

### For ALL Projects:
- [ ] GitHub repository with meaningful README
- [ ] Clean, modular code (functions, classes, modules)
- [ ] Type hints throughout
- [ ] Docstrings for all functions/classes
- [ ] requirements.txt or pyproject.toml
- [ ] At least basic tests
- [ ] No hardcoded paths (use config or constants)
- [ ] Error handling (try/except where appropriate)
- [ ] Logging configured
- [ ] .gitignore file

### For ML-Focused Projects (1, 3, 4):
- [ ] Reproducible with fixed random seed
- [ ] Train/test split (or cross-validation)
- [ ] Multiple models compared
- [ ] Evaluation metrics clearly reported
- [ ] Feature importance or model interpretation

### For Engineering-Focused Projects (2, 5):
- [ ] Dockerfile with multi-stage build
- [ ] docker-compose.yml (if multi-service)
- [ ] CI/CD configuration
- [ ] Health check endpoint
- [ ] Performance benchmark

## Final Advice

1. **Start simple**: Get a minimal version working first, then add features
2. **Commit early, commit often**: Use Git from the start
3. **Test incrementally**: Write tests as you go, not at the end
4. **Document decisions**: Comment WHY not just WHAT
5. **Read the docs**: Official docs for sklearn, FastAPI, Docker are excellent
6. **Ask for feedback**: Share your architecture before diving deep
7. **Polish the README**: This is what people see first in your portfolio